# BirdCLEF 2026 — Pseudo-Label Spectrogram Generation Round 2 (Part 2 of 2)

Generates 224×224 mel-spectrogram PNGs for the second half of unlabeled train soundscapes
(files 5296–10591), filtered to clips where max ensemble probability ≥ 0.5.

Uses Round 2 predictions from the V3+V9 ensemble (0.811).

Outputs a flat folder of PNGs + `pseudo_label_index.csv`, zipped in batches to stay
within disk limits, then uploaded to the `pseudo-labels-part2-r2` Kaggle dataset.

In [ ]:
from __future__ import annotations
import json
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# --- mel spectrogram params (v1 — matches original ensemble training data) ---
TARGET_SIZE    = (224, 224)
SAMPLE_RATE    = 32000
CLIP_DURATION  = 5
STRIDE_DURATION = 2.5
HOP_LENGTH     = 512   # librosa defaults for everything else

# --- filtering ---
CONFIDENCE_THRESHOLD = 0.5

# --- file range for this notebook (0-indexed into the sorted unlabeled list) ---
FILE_START = 5296
FILE_END   = None   # None = process to end of list

# --- batch size: soundscapes per zip ---
BATCH_SIZE = 100

# --- paths ---
PARQUET_PATH = Path('/kaggle/input/notebooks/ucheozoemena/birdclef-2026-pseudo-label-full-inference-r2/pseudo_label_predictions.parquet')
WORK_ROOT    = Path('/kaggle/working/work')
ZIP_STAGING  = Path('/kaggle/working/upload')
INDEX_PATH   = ZIP_STAGING / 'pseudo_label_index.csv'
ERROR_LOG    = Path('/kaggle/working/errors.log')

# --- target Kaggle dataset ---
DATASET_ID    = 'ucheozoemena/pseudo-labels-part2-r2'
DATASET_TITLE = 'BirdCLEF 2026 Pseudo Labels Part 2 R2'

In [ ]:
import kagglehub
competition_dir = Path(kagglehub.competition_download('birdclef-2026'))
print('Competition dir:', competition_dir)

labels_df = pd.read_csv(competition_dir / 'train_soundscapes_labels.csv')
labeled_files = set(labels_df['filename'].unique())

all_soundscapes = sorted((competition_dir / 'train_soundscapes').glob('*.ogg'))
unlabeled = [f for f in all_soundscapes if f.name not in labeled_files]
subset = unlabeled[FILE_START:FILE_END]
print(f'Total unlabeled soundscapes: {len(unlabeled)}')
print(f'This notebook processes files {FILE_START}–end: {len(subset)} files')

In [ ]:
preds_df = pd.read_parquet(PARQUET_PATH)
species_cols = list(preds_df.columns[3:])
print(f'Parquet loaded: {len(preds_df):,} rows, {len(species_cols)} species')

preds_by_file = {fname: grp for fname, grp in preds_df.groupby('filename')}
print(f'Unique files in parquet: {len(preds_by_file)}')

In [ ]:
WORK_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_STAGING.mkdir(parents=True, exist_ok=True)
if ERROR_LOG.exists():
    ERROR_LOG.unlink()

clip_length    = int(CLIP_DURATION * SAMPLE_RATE)
stride_samples = int(STRIDE_DURATION * SAMPLE_RATE)
stride_frames  = int(STRIDE_DURATION * SAMPLE_RATE / HOP_LENGTH)
frames_per_clip = int(CLIP_DURATION * SAMPLE_RATE / HOP_LENGTH)

def normalize_to_uint8(s_db):
    lo, hi = float(s_db.min()), float(s_db.max())
    if hi == lo:
        return np.zeros_like(s_db, dtype=np.uint8)
    return ((s_db - lo) / (hi - lo) * 255).astype(np.uint8)

def zip_dir(source_dir, zip_path):
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(source_dir.rglob('*')):
            if fp.is_file():
                zf.write(fp, arcname=fp.relative_to(source_dir))

index_rows = []
total_pngs = 0
total_skipped = 0

n_total = len(subset)

for batch_start in range(0, n_total, BATCH_SIZE):
    batch_files = subset[batch_start:batch_start + BATCH_SIZE]
    batch_num   = batch_start // BATCH_SIZE + 1
    batch_slug  = f'batch_{batch_num:04d}'
    batch_work  = WORK_ROOT / batch_slug
    batch_zip   = ZIP_STAGING / f'{batch_slug}.zip'

    if batch_zip.exists():
        print(f'[{batch_slug}] zip exists, skipping.')
        continue

    if batch_work.exists():
        shutil.rmtree(batch_work)
    batch_work.mkdir(parents=True)

    batch_pngs = 0
    for soundscape in batch_files:
        if soundscape.name not in preds_by_file:
            total_skipped += 1
            continue
        file_preds = preds_by_file[soundscape.name]

        try:
            samples, _ = librosa.load(soundscape, sr=SAMPLE_RATE)
            S_db = librosa.power_to_db(
                librosa.feature.melspectrogram(y=samples, sr=SAMPLE_RATE, hop_length=HOP_LENGTH),
                ref=np.max,
            )
            stem = soundscape.stem

            for _, row in file_preds.iterrows():
                probs = row[species_cols].values.astype(np.float32)
                if probs.max() < CONFIDENCE_THRESHOLD:
                    continue

                k = int(row['clip_index'])
                window = S_db[:, k * stride_frames : k * stride_frames + frames_per_clip]
                if window.shape[1] < frames_per_clip:
                    continue

                img = Image.fromarray(normalize_to_uint8(window)).resize(TARGET_SIZE).convert('RGB')
                png_name = f'{stem}__k{k}.png'
                img.save(batch_work / png_name)

                present = [species_cols[j] for j, p in enumerate(probs) if p >= CONFIDENCE_THRESHOLD]
                index_rows.append({'image_path': png_name, 'labels': ';'.join(present)})
                batch_pngs += 1

        except Exception as exc:
            with ERROR_LOG.open('a') as f:
                f.write(f'{soundscape}\t{exc}\n')

    zip_dir(batch_work, batch_zip)
    mb = batch_zip.stat().st_size / 1e6
    shutil.rmtree(batch_work)
    total_pngs += batch_pngs
    global_end = min(batch_start + BATCH_SIZE, n_total)
    print(f'[{batch_slug}] files {FILE_START+batch_start}–{FILE_START+global_end-1}: '
          f'{batch_pngs} PNGs → {batch_zip.name} ({mb:.0f} MB)')

print(f'\nDone. Total PNGs: {total_pngs:,}  Skipped files: {total_skipped}')
if ERROR_LOG.exists():
    print(f'Errors: {sum(1 for _ in ERROR_LOG.open())} (see {ERROR_LOG})')

In [ ]:
index_df = pd.DataFrame(index_rows)
index_df.to_csv(INDEX_PATH, index=False)
print(f'Index saved: {len(index_df):,} rows → {INDEX_PATH}')
print(index_df.head())

In [ ]:
(ZIP_STAGING / 'dataset-metadata.json').write_text(json.dumps({
    'title': DATASET_TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}, indent=2))

zips = sorted(ZIP_STAGING.glob('*.zip'))
total_mb = sum(z.stat().st_size for z in zips) / 1e6
print(f'Uploading {len(zips)} zip(s) + index CSV, {total_mb:.0f} MB total')

result = subprocess.run(
    ['kaggle', 'datasets', 'version', '-p', str(ZIP_STAGING), '-m', 'initial upload'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

if result.returncode != 0:
    print('version failed — trying create...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', str(ZIP_STAGING)],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)

print('Upload complete!' if result.returncode == 0 else f'Upload failed (exit {result.returncode})')